# コードレビュー変更点の影響を特定する実験

コードレビューで加えた主要な変更（I2・I4）の単独・複合効果を測定する。

## テストする4バリアント（rounds=300固定）

| バリアント | bfill (I2) | Absolute_Meeting_ID<8除外 (I4) | 説明 |
|-----------|-----------|-------------------------------|------|
| A (旧)    | あり       | なし                           | コードレビュー前の状態 |
| B         | あり       | あり                           | I4のみ適用 |
| C         | なし       | なし                           | I2のみ適用 |
| D (新)    | なし       | あり                           | 現行コード（I2+I4両方） |

**読み方**: D（現行）が最も低くなるはず。AとBの差→I4の影響、AとCの差→I2の影響。

In [1]:
import sys
sys.path.insert(0, '..')

import bisect
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

from src.features import generate_features
from src.modeling import walk_forward_validation

EXCEL_PATH   = '../data/BOJ_data.xlsx'
MEETING_PATH = '../data/BOJ_meeting_history.csv'
START_DATE   = '2024-01-01'
ROUNDS       = 300

In [2]:
# ===== パラメータ付きパイプライン関数 =====

def load_data(excel_path, meeting_csv_path, use_bfill=True):
    """use_bfill=True → 旧コード、use_bfill=False → 新コード（I2適用）"""
    df_raw = pd.read_excel(excel_path)
    df = df_raw.iloc[1:].copy()
    df['日付'] = pd.to_datetime(df['日付'], format='%Y年%m月%d日')
    df = df.sort_values('日付').reset_index(drop=True)

    rename_dict = {
        'JPBOJ1ONI=TRDT (MID_PRICE)': 'M1', 'JPBOJ2ONI=TRDT (MID_PRICE)': 'M2',
        'JPBOJ3ONI=TRDT (MID_PRICE)': 'M3', 'JPBOJ4ONI=TRDT (MID_PRICE)': 'M4',
        'JPBOJ5ONI=TRDT (MID_PRICE)': 'M5', 'JPBOJ6ONI=TRDT (MID_PRICE)': 'M6',
        'JPBOJ7ONI=TRDT (MID_PRICE)': 'M7', 'JPBOJ8ONI=TRDT (MID_PRICE)': 'M8',
        'JPY= (MID_PRICE)': 'USDJPY', 'JGBc1 (TRDPRC_1)': 'JGB_Future',
        '.N225 (TRDPRC_1)': 'Nikkei225', '.DXY (TRDPRC_1)': 'DXY',
        'JP12MONI=TRDT (BID)': 'T12', 'JP18MONI=TRDT (BID)': 'T18', 'JP24MONI=TRDT (BID)': 'T24',
    }
    df = df.rename(columns=rename_dict)
    cols_to_keep = ['日付','M1','M2','M3','M4','M5','M6','M7','M8',
                    'USDJPY','JGB_Future','Nikkei225','DXY','T12','T18','T24']
    df = df[[c for c in cols_to_keep if c in df.columns]].copy()
    for col in df.columns.drop('日付'):
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df_meetings = pd.read_csv(meeting_csv_path)
    df_meetings['Date'] = pd.to_datetime(df_meetings['Date'])
    df = pd.merge(df, df_meetings[['Date','Policy_Rate','Event']],
                  left_on='日付', right_on='Date', how='left', suffixes=('','_mtg'))
    df = df.drop(columns=['Date']).rename(columns={'日付': 'Date'})
    df['Is_Meeting_Day'] = df['Event'].notnull().astype(int)

    if use_bfill:
        df['Actual_Policy_Rate'] = df['Policy_Rate'].ffill().bfill()  # 旧
    else:
        df['Actual_Policy_Rate'] = df['Policy_Rate'].ffill()           # 新（I2）

    all_rate_cols = [c for c in ['M1','M2','M3','M4','M5','M6','M7','M8','T12','T18','T24'] if c in df.columns]
    for col in all_rate_cols:
        df[f'{col}_is_imputed'] = df[col].isnull().astype(int)

    impute_cols = all_rate_cols + [c for c in ['USDJPY','JGB_Future','Nikkei225','DXY'] if c in df.columns]
    imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=42)
    df[impute_cols] = imputer.fit_transform(df[impute_cols])

    all_meeting_dates = sorted(df_meetings['Date'].unique())
    def days_to_next_mpm(date):
        idx = bisect.bisect_left(all_meeting_dates, date)
        return (all_meeting_dates[idx] - date).days if idx < len(all_meeting_dates) else np.nan
    df['Days_to_MPM'] = df['Date'].map({d: days_to_next_mpm(d) for d in df['Date'].unique()})

    tenor_cols = [c for c in ['T12','T18','T24'] if c in df.columns]
    ext_cols   = [c for c in ['USDJPY','JGB_Future','Nikkei225','DXY'] if c in df.columns]
    final_cols = (['Date','M1','M2','M3','M4','M5','M6','M7','M8'] + tenor_cols + ext_cols +
                  ['Actual_Policy_Rate','Is_Meeting_Day','Days_to_MPM'] +
                  [f'{c}_is_imputed' for c in all_rate_cols])
    return df[final_cols]


def pool_data(df, exclude_low_abs_id=True):
    """exclude_low_abs_id=False → 旧コード、True → 新コード（I4適用）"""
    boj_rate_cols   = [f'M{i}' for i in range(1, 9)]
    tenor_rate_cols = [c for c in ['T12','T18','T24'] if c in df.columns]
    raw_rate_cols   = boj_rate_cols + tenor_rate_cols
    id_cols = [col for col in df.columns if col not in raw_rate_cols]

    pooled_boj = df.melt(id_vars=id_cols, value_vars=boj_rate_cols,
                         var_name='Rate_Label', value_name='Rate_Value')
    pooled_boj['Meeting_Index'] = pooled_boj['Rate_Label'].str.extract(r'(\d+)').astype(int)
    pooled_boj['Is_Tenor_OIS']  = 0

    tenor_index_map = {'T12': 10, 'T18': 11, 'T24': 12}
    if tenor_rate_cols:
        pooled_tenor = df.melt(id_vars=id_cols, value_vars=tenor_rate_cols,
                               var_name='Rate_Label', value_name='Rate_Value')
        pooled_tenor['Meeting_Index'] = pooled_tenor['Rate_Label'].map(tenor_index_map)
        pooled_tenor['Is_Tenor_OIS']  = 1
        pooled = pd.concat([pooled_boj, pooled_tenor], ignore_index=True)
    else:
        pooled = pooled_boj

    pooled = pooled.sort_values(['Rate_Label', 'Date']).reset_index(drop=True)

    df_dates = df[['Date','Days_to_MPM']].drop_duplicates().dropna(subset=['Days_to_MPM'])
    df_dates['Next_Meeting_Date'] = df_dates['Date'] + pd.to_timedelta(df_dates['Days_to_MPM'].astype(int), unit='D')
    all_meeting_dates = sorted(df_dates['Next_Meeting_Date'].unique())
    meeting_date_to_rank = {m: i for i, m in enumerate(all_meeting_dates)}
    date_to_next_rank = dict(zip(df_dates['Date'], df_dates['Next_Meeting_Date'].map(meeting_date_to_rank)))

    pooled['_next_rank'] = pooled['Date'].map(date_to_next_rank)
    pooled['Absolute_Meeting_ID'] = (pooled['_next_rank'] + pooled['Meeting_Index'] - 1).astype('Int64')
    pooled.loc[pooled['Is_Tenor_OIS'] == 1, 'Absolute_Meeting_ID'] = pd.NA
    pooled = pooled.drop(columns=['_next_rank'])

    data_start_date = df['Date'].min()
    def get_first_seen_date(abs_id):
        if pd.isna(abs_id): return pd.NaT
        k = int(abs_id)
        pred = k - 8
        if pred < 0: return data_start_date
        elif pred < len(all_meeting_dates): return all_meeting_dates[pred]
        else: return pd.NaT

    abs_id_to_first_seen = {k: get_first_seen_date(k) for k in pooled['Absolute_Meeting_ID'].dropna().unique()}
    pooled['First_Seen_Date'] = pooled['Absolute_Meeting_ID'].map(abs_id_to_first_seen)
    pooled['Days_since_first_seen'] = (pooled['Date'] - pooled['First_Seen_Date']).dt.days
    pooled = pooled.drop(columns=['First_Seen_Date'])

    if exclude_low_abs_id:  # I4適用
        mask = (pooled['Is_Tenor_OIS'] == 0) & (pooled['Absolute_Meeting_ID'] < 8)
        pooled = pooled[~mask].reset_index(drop=True)

    for h in [1, 3, 5]:
        pooled[f'Target_{h}d'] = pooled.groupby('Rate_Label')['Rate_Value'].shift(-h) - pooled['Rate_Value']
    for h in [3, 5]:
        instr_std = pooled.groupby('Rate_Label')[f'Target_{h}d'].transform('std')
        pooled[f'Target_{h}d_std']  = instr_std
        pooled[f'Target_{h}d_norm'] = pooled[f'Target_{h}d'] / instr_std

    df_date_meeting = df[['Date','Is_Meeting_Day']].copy()
    df_date_meeting['is_post_mpm'] = (
        df_date_meeting['Is_Meeting_Day'].rolling(window=6, min_periods=1).max() == 1
    ).astype(int)
    pooled = pd.merge(pooled, df_date_meeting[['Date','is_post_mpm']], on='Date', how='left')

    boj_spread_cols   = [f'M{i}_spread'     for i in range(1, 9)]
    boj_fd_cols       = [f'M{i}_frac_diff'  for i in range(1, 9)]
    boj_imp_cols      = [f'M{i}_is_imputed' for i in range(1, 9)]
    tenor_spread_cols = [f'{c}_spread'     for c in tenor_rate_cols]
    tenor_fd_cols     = [f'{c}_frac_diff'  for c in tenor_rate_cols]
    tenor_imp_cols    = [f'{c}_is_imputed' for c in tenor_rate_cols]
    ext_fd_cols       = ['USDJPY_frac_diff','JGB_Future_frac_diff','Nikkei225_frac_diff']
    basic_cols  = ['Date','Rate_Label','Meeting_Index','Is_Tenor_OIS','is_post_mpm',
                   'Days_to_MPM','Actual_Policy_Rate','Absolute_Meeting_ID','Days_since_first_seen']
    target_cols = ['Target_1d','Target_3d','Target_3d_norm','Target_3d_std',
                   'Target_5d','Target_5d_norm','Target_5d_std']
    curve_cols    = ['Curve_Slope'] + [f'Butterfly_M{n}' for n in range(2, 8)]
    curve_fd_cols = [f'{c}_frac_diff' for c in curve_cols]
    cyclic_cols   = ['DayOfWeek_sin','DayOfWeek_cos','Days_to_MPM_sin','Days_to_MPM_cos']
    final_cols = (basic_cols + boj_spread_cols + boj_fd_cols + boj_imp_cols
                  + tenor_spread_cols + tenor_fd_cols + tenor_imp_cols
                  + ext_fd_cols + curve_cols + curve_fd_cols + cyclic_cols + target_cols)
    return pooled[[c for c in final_cols if c in pooled.columns]]

In [3]:
# 4バリアントのデータを準備
variants = [
    {'name': 'A (旧: bfill=あり, I4=なし)',   'use_bfill': True,  'exclude_i4': False},
    {'name': 'B (I4のみ: bfill=あり, I4=あり)', 'use_bfill': True,  'exclude_i4': True},
    {'name': 'C (I2のみ: bfill=なし, I4=なし)', 'use_bfill': False, 'exclude_i4': False},
    {'name': 'D (新: bfill=なし, I4=あり)',    'use_bfill': False, 'exclude_i4': True},
]

results = []

for v in variants:
    print(f"バリアント {v['name']} を実行中...")

    df_raw  = load_data(EXCEL_PATH, MEETING_PATH, use_bfill=v['use_bfill'])
    df_feat = generate_features(df_raw)
    df_pool = pool_data(df_feat, exclude_low_abs_id=v['exclude_i4'])

    n_rows = len(df_pool)

    res_3d = walk_forward_validation(df_pool, 'Target_3d_norm', START_DATE, num_boost_round=ROUNDS)
    res_5d = walk_forward_validation(df_pool, 'Target_5d_norm', START_DATE, num_boost_round=ROUNDS)

    ic_3d, _ = spearmanr(res_3d['Actual'], res_3d['Pred'])
    ic_5d, _ = spearmanr(res_5d['Actual'], res_5d['Pred'])
    train_ic_3d = res_3d['Train_IC'].mean()
    train_ic_5d = res_5d['Train_IC'].mean()

    results.append({
        'バリアント':   v['name'],
        'データ行数':   n_rows,
        'OOS_IC_3d':   round(ic_3d, 4),
        'OOS_IC_5d':   round(ic_5d, 4),
        'Train_IC_3d': round(train_ic_3d, 4),
        'Train_IC_5d': round(train_ic_5d, 4),
        'Gap_3d':      round(train_ic_3d - ic_3d, 4),
        'Gap_5d':      round(train_ic_5d - ic_5d, 4),
    })
    print(f"  → 3d IC={ic_3d:.4f}, 5d IC={ic_5d:.4f}, 行数={n_rows:,}")

df_iso = pd.DataFrame(results)
print('\n=== 変更点の影響 分離実験結果（rounds=300） ===')
print(df_iso.to_string(index=False))

バリアント A (旧: bfill=あり, I4=なし) を実行中...


  → 3d IC=0.2277, 5d IC=0.2178, 行数=51,183
バリアント B (I4のみ: bfill=あり, I4=あり) を実行中...


  → 3d IC=0.1864, 5d IC=0.1889, 行数=21,144
バリアント C (I2のみ: bfill=なし, I4=なし) を実行中...


  → 3d IC=0.1873, 5d IC=0.1931, 行数=51,183
バリアント D (新: bfill=なし, I4=あり) を実行中...


  → 3d IC=0.1814, 5d IC=0.1908, 行数=21,144

=== 変更点の影響 分離実験結果（rounds=300） ===
                    バリアント  データ行数  OOS_IC_3d  OOS_IC_5d  Train_IC_3d  Train_IC_5d  Gap_3d  Gap_5d
   A (旧: bfill=あり, I4=なし)  51183     0.2277     0.2178       0.6091       0.6737  0.3815  0.4558
B (I4のみ: bfill=あり, I4=あり)  21144     0.1864     0.1889       0.8052       0.8309  0.6188  0.6419
C (I2のみ: bfill=なし, I4=なし)  51183     0.1873     0.1931       0.8116       0.8296  0.6243  0.6365
   D (新: bfill=なし, I4=あり)  21144     0.1814     0.1908       0.8066       0.8270  0.6252  0.6362


In [4]:
# 影響の整理
a = df_iso[df_iso['バリアント'].str.startswith('A')].iloc[0]
b = df_iso[df_iso['バリアント'].str.startswith('B')].iloc[0]
c = df_iso[df_iso['バリアント'].str.startswith('C')].iloc[0]
d = df_iso[df_iso['バリアント'].str.startswith('D')].iloc[0]

print('=== I2（bfill削除）の単独影響: A→C ===')
print(f'  3d IC変化: {a["OOS_IC_3d"]:.4f} → {c["OOS_IC_3d"]:.4f}  (Δ={c["OOS_IC_3d"]-a["OOS_IC_3d"]:+.4f})')
print(f'  5d IC変化: {a["OOS_IC_5d"]:.4f} → {c["OOS_IC_5d"]:.4f}  (Δ={c["OOS_IC_5d"]-a["OOS_IC_5d"]:+.4f})')

print('\n=== I4（Absolute_Meeting_ID<8除外）の単独影響: A→B ===')
print(f'  3d IC変化: {a["OOS_IC_3d"]:.4f} → {b["OOS_IC_3d"]:.4f}  (Δ={b["OOS_IC_3d"]-a["OOS_IC_3d"]:+.4f})')
print(f'  5d IC変化: {a["OOS_IC_5d"]:.4f} → {b["OOS_IC_5d"]:.4f}  (Δ={b["OOS_IC_5d"]-a["OOS_IC_5d"]:+.4f})')

print('\n=== I2+I4 複合影響: A→D ===')
print(f'  3d IC変化: {a["OOS_IC_3d"]:.4f} → {d["OOS_IC_3d"]:.4f}  (Δ={d["OOS_IC_3d"]-a["OOS_IC_3d"]:+.4f})')
print(f'  5d IC変化: {a["OOS_IC_5d"]:.4f} → {d["OOS_IC_5d"]:.4f}  (Δ={d["OOS_IC_5d"]-a["OOS_IC_5d"]:+.4f})')

=== I2（bfill削除）の単独影響: A→C ===
  3d IC変化: 0.2277 → 0.1873  (Δ=-0.0404)
  5d IC変化: 0.2178 → 0.1931  (Δ=-0.0247)

=== I4（Absolute_Meeting_ID<8除外）の単独影響: A→B ===
  3d IC変化: 0.2277 → 0.1864  (Δ=-0.0413)
  5d IC変化: 0.2178 → 0.1889  (Δ=-0.0289)

=== I2+I4 複合影響: A→D ===
  3d IC変化: 0.2277 → 0.1814  (Δ=-0.0463)
  5d IC変化: 0.2178 → 0.1908  (Δ=-0.0270)
